<a href="https://colab.research.google.com/github/natthamonsmn/Cinema_Team/blob/main/Cinema.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍿🎥✮⋆˙ ธุรกิจโรงภาพยนตร์ (Cinema)
เน้น **ระบบจองที่นั่งดูหนัง** เช่น การจองที่นั่งของลูกค้า, ตรวจสอบที่นั่งว่าง, คำนวณราคาตามประเภทที่นั่งและประเภทสมาชิก

## 1. โครงสร้างโปรแกรม

## ส่วนที่ 1 — เตรียม class และฟังก์ชัน

In [ ]:
from datetime import datetime, timedelta
import random
import pandas as pd
import numpy as np
import os, shutil
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

In [ ]:
!pip install Faker
from faker import Faker #สุ่มชื่อภาษาไทยสมจริงอัตโนมัติ
fake = Faker("th_TH")

In [ ]:
# 1. ติดตั้งฟอนต์ภาษาไทย
!apt-get -y install fonts-thai-tlwg

# 2. ล้าง cache ของ matplotlib เพื่ออัปเดตฟอนต์ใหม่
cache_dir = matplotlib.get_cachedir()
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

# 3. ลงทะเบียนฟอนต์ใหม่เข้ากับ FontManager
font_path = '/usr/share/fonts/truetype/tlwg/Loma.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)

# 4. ตั้งค่าฟอนต์หลัก
plt.rcParams['font.family'] = 'Loma'
plt.rcParams['axes.unicode_minus'] = False

# 5. สร้างข้อมูลจำลองและแสดงผล
data = {
    'เมือง': ['กรุงเทพฯ', 'เชียงใหม่', 'ภูเก็ต'],
    'อุณหภูมิ': [30.5, 25.2, 28.9]
}

df_thai = pd.DataFrame(data)

print("ตั้งค่าระบบฟอนต์ภาษาไทยสำเร็จ!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-thai-tlwg is already the newest version (1:0.7.3-1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
ตั้งค่าระบบฟอนต์ภาษาไทยสำเร็จ!


**Class** ของระบบการจองตั๋วโรงภาพยนตร์ ประกอบด้วย 3 class ดังนี้ :

| Class | Object | Attributes | Methods / Functions | |
|---|---|---|---|---|
| `Customer` | เก็บข้อมูลของผู้ใช้งานและสิทธิ์ส่วนลด | รหัสลูกค้า, ชื่อ-นามสกุล, ระดับสมาชิก, แต้มสะสม|  ตรวจสอบสิทธิ์ส่วนลดตามระดับสมาชิก, สะสมแต้มจากการจองสำเร็จ |
| `Showtime` | เก็บข้อมูลหนัง วันเวลาที่ฉาย และราคาพื้นฐาน|  รหัสรอบฉาย, ชื่อภาพยนตร์, หมายเลขโรงภาพยนตร์ , วันและเวลาที่ฉาย, ราคาพื้นฐานของรอบฉาย| ตรวจสอบไม่ให้โรงหนังเดียวกันมีรอบฉายเวลาชนกัน |
| `Booking` |  เชื่อมระหว่าง "ลูกค้า" และ "รอบฉาย" เพื่อบันทึกรายละเอียดการจอง | รหัสการจอง, รหัสลูกค้า, รหัสรอบฉาย, เลขที่นั่ง, ประเภทที่นั่ง, ราคาสุทธิ, สถานะการจอง| คำนวณราคาสุทธิ, ตรวจสอบว่าที่นั่งในรอบนั้นยังว่างอยู่หรือไม่, ยืนยันและบันทึกการจองลงระบบ |

In [ ]:
class Customer:
    def __init__(self, customer_id, name, member_type="General"):
        self.customer_id = customer_id
        self.name = name
        self.member_type = member_type
        self.points = 0

    def get_discount_rate(self, promo_discount_rate=0.0):
        member_discounts = {"General": 0.0, "Student": 0.20, "VIP": 0.30}
        member_rate = member_discounts.get(self.member_type, 0.0)
        return max(member_rate, promo_discount_rate)

    def add_points(self, net_price):
        earned = int(net_price // 10)
        self.points += earned
        return earned


class Showtime:
    def __init__(self, showtime_id, movie_name, theater_no, start_time, end_time, base_price):
        self.showtime_id = showtime_id
        self.movie_name = movie_name
        self.theater_no = theater_no
        self.start_time = start_time
        self.end_time = end_time
        self.base_price = base_price


class Booking:
    def __init__(self, booking_id, customer, showtime, seat_number, seat_type):
        self.booking_id = booking_id
        self.customer = customer
        self.showtime = showtime
        self.seat_number = seat_number.upper()
        self.seat_type = seat_type
        self.booking_time = datetime.now()
        self.status = "Pending"
        # เรียกใช้ Helper Function คิดราคา
        self.net_price = calculate_booking_price(showtime.base_price, seat_type, customer.member_type)

    def check_timeout(self):
        if self.status == "Pending":
            if datetime.now() > self.booking_time + timedelta(minutes=10):
                self.status = "Cancelled"
                return True
        return False

    def confirm_payment(self):
        if self.check_timeout():
            raise Exception("เกินเวลา 10 นาที (Timeout Auto-Cancel)")
        self.status = "Paid"
        self.customer.add_points(self.net_price)
        return self.net_price

## ฟังก์ชันช่วยงาน (Helper Function)

In [ ]:
def calculate_booking_price(base_price, seat_type="Normal", member_type="General", promo_discount=0.0):
    """ฟังก์ชันที่ 1: คำนวณราคาสุทธิ (มี Default Argument)"""
    seat_extras = {"Normal": 0, "Deluxe": 40, "Honeymoon": 80, "VIP": 190}
    member_discounts = {"General": 0.0, "Student": 0.20, "VIP": 0.30}

    total_seat_price = base_price + seat_extras.get(seat_type, 0)
    best_discount = max(member_discounts.get(member_type, 0.0), promo_discount)
    return round(total_seat_price * (1 - best_discount), 2)


def validate_seat_booking(seat_number, existing_seats_list, valid_seat_range=None):
    """ฟังก์ชันที่ 2: ตรวจสอบผังและที่นั่งซ้ำ (มี Default Argument)"""
    if valid_seat_range is None:
        valid_seat_range = [f"{row}{num}" for row in "ABCDEF" for num in range(1, 11)]

    seat_upper = seat_number.upper()
    if seat_upper not in valid_seat_range:
        return False, f"ที่นั่ง {seat_number} ไม่มีอยู่ในระบบ (Invalid Seat)"
    if seat_upper in existing_seats_list:
        return False, f"ที่นั่ง {seat_number} ถูกจองแล้ว (Double Booking)"
    return True, "ที่นั่งว่าง สามารถจองได้"


def process_ticket_scan(ticket_id, current_status="Paid", auto_update=True):
    """ฟังก์ชันที่ 3: จำลองการสแกน QR Code เข้าโรง (มี Default Argument)"""
    if current_status == "Used":
        return False, f"ตั๋ว {ticket_id} ถูกใช้งานไปแล้ว ไม่สามารถสแกนซ้ำได้!"
    elif current_status != "Paid":
        return False, f"ตั๋ว {ticket_id} ยังไม่ได้ชำระเงิน"

    new_status = "Used" if auto_update else current_status
    return True, f"สแกนสำเร็จ! ({new_status})"

def mask_lastname_only(fullname):
    parts = fullname.split()
    if len(parts) >= 2:
        first_name, last_name = parts[0], parts[1]
        # ชื่อจริงเต็ม + อักษรแรกนามสกุล แล้วตามด้วย ***
        return f"{first_name} {last_name[0]}***"
    return fullname

## จำลอง 300 รายการทีละคน

In [ ]:
# Execution Simulation Loop (จำลอง 300 รายการทีละคน)
showtime_1 = Showtime(
    "ST01",
    "Avatar 3",
    1,
    datetime(2026, 8, 20, 14, 0),
    datetime(2026, 8, 20, 17, 0),
    160,
)
showtime_2 = Showtime(
    "ST02",
    "Spider-Man",
    2,
    datetime(2026, 8, 20, 15, 30),
    datetime(2026, 8, 20, 18, 0),
    180,
)
showtime_3 = Showtime(
    "ST03",
    "Batman",
    3,
    datetime(2026, 8, 20, 19, 0),
    datetime(2026, 8, 20, 22, 0),
    200,
)

showtime_4 = Showtime(
    "ST04",
    "Inception",
    4,
    datetime(2026, 8, 20, 13, 0),
    datetime(2026, 8, 20, 15, 30),
    170,
)
showtime_5 = Showtime(
    "ST05",
    "Interstellar",
    5,
    datetime(2026, 8, 20, 20, 0),
    datetime(2026, 8, 20, 23, 0),
    190,
)

showtimes_pool = [
    showtime_1,
    showtime_2,
    showtime_3,
    showtime_4,
    showtime_5,
]

# ป้องกันการจองซ้ำ
booked_seats_by_theater = {1: [], 2: [], 3: [], 4: [], 5: []}

all_bookings = []
member_pool = ["General", "Student", "VIP"]
seat_type_pool = ["Normal", "Deluxe", "Honeymoon", "VIP"]
seat_pool = [f"{r}{n}" for r in "ABCDEF" for n in range(1, 11)] + ["Z99"]

# Loop จำลอง 300 รายการ (สุ่มเลือกโรงภาพยนตร์)
first_names = [
    "สมชาย",
    "วิภา",
    "กิตติ",
    "อนันต์",
    "นภา",
    "กมล",
    "ธนกฤต",
    "ศิริพร",
    "ชลธิชา",
    "ณัฐพงษ์",
]
last_names = [
    "ใจดี",
    "รักสงบ",
    "เจริญสุข",
    "มีมาก",
    "มั่นคง",
    "วงศ์สวัสดิ์",
    "สมบูรณ์",
    "สุวรรณ",
]

for i in range(1, 301):
    cust_id = f"C{i:03d}"
    cust_name = f"{random.choice(first_names)} {random.choice(last_names)}"

    chosen_member = random.choice(member_pool)
    chosen_seat = random.choice(seat_pool)
    chosen_seat_type = random.choice(seat_type_pool)
    booking_id = f"BK{i:03d}"

    # สุ่มเลือกโรงภาพยนตร์ 1 จาก 3 โรง
    chosen_showtime = random.choice(showtimes_pool)
    theater_no = chosen_showtime.theater_no

    customer = Customer(cust_id, cust_name, chosen_member)

    # ตรวจสอบที่นั่งซ้ำเฉพาะในโรงภาพยนตร์ที่เลือก
    is_valid, seat_msg = validate_seat_booking(
        chosen_seat, booked_seats_by_theater[theater_no]
    )

    if not is_valid:
        failed_booking = Booking(
            booking_id,
            customer,
            chosen_showtime,
            chosen_seat,
            chosen_seat_type,
        )
        failed_booking.status = f"Rejected ({seat_msg})"
        failed_booking.net_price = 0.0
        all_bookings.append(failed_booking)
        continue

    try:
        new_booking = Booking(
            booking_id,
            customer,
            chosen_showtime,
            chosen_seat,
            chosen_seat_type,
        )

        if random.random() < 0.10:
            new_booking.booking_time = datetime.now() - timedelta(minutes=15)

        new_booking.confirm_payment()
        booked_seats_by_theater[theater_no].append(chosen_seat)
        all_bookings.append(new_booking)

    except Exception:
        new_booking.net_price = 0.0
        all_bookings.append(new_booking)

## ส่วนที่ 2 — ทดสอบฟังก์ชันทีละตัว

In [ ]:
seat_pool = [f"{r}{n}" for r in "ABCDEF" for n in range(1, 11)]
# ทดสอบเรียกสุ่มที่นั่ง 3 ครั้ง ดูว่าฟังก์ชันสุ่มทำงานได้ค่าไม่ซ้ำกันในแต่ละรอบ
for _ in range(3):
    print("ที่นั่งที่สุ่มได้:", random.choice(seat_pool))

ที่นั่งที่สุ่มได้: C7
ที่นั่งที่สุ่มได้: A1
ที่นั่งที่สุ่มได้: D9


In [ ]:
booked_test_list = ["A1"]

# ทดสอบการตรวจที่นั่ง (ที่นั่งซ้ำ, ที่นั่งว่าง, ที่นั่งไม่มีอยู่จริง)
for seat_to_check in ["A1", "B5", "Z99"]:
    is_valid, message = validate_seat_booking(seat_to_check, booked_test_list)
    print(f"ผลการตรวจที่นั่ง '{seat_to_check}':", message)

ผลการตรวจที่นั่ง 'A1': ที่นั่ง A1 ถูกจองแล้ว (Double Booking)
ผลการตรวจที่นั่ง 'B5': ที่นั่งว่าง สามารถจองได้
ผลการตรวจที่นั่ง 'Z99': ที่นั่ง Z99 ไม่มีอยู่ในระบบ (Invalid Seat)


In [ ]:
# เปรียบเทียบการเรียก calculate_booking_price() แบบไม่ระบุค่า (ใช้ default) กับแบบระบุช่วงเอง
print("เรียกแบบ default (ไม่ใส่ argument):", calculate_booking_price(160))
print(
    "เรียกแบบระบุช่วงเอง (ที่นั่ง VIP, สมาชิก VIP):",
    calculate_booking_price(160, "VIP", "VIP"),
)
print(
    "เรียกแบบระบุแค่ promo_discount (keyword argument):",
    calculate_booking_price(160, promo_discount=0.50),
)

เรียกแบบ default (ไม่ใส่ argument): 160.0
เรียกแบบระบุช่วงเอง (ที่นั่ง VIP, สมาชิก VIP): 245.0
เรียกแบบระบุแค่ promo_discount (keyword argument): 80.0


## ส่วนที่ 3 — ฟังก์ชันอธิบายขั้นตอนคำนวณราคา

In [ ]:
def explain_booking_price(booking):
    """รับ object Booking 1 ตัว -> พิมพ์อธิบายการคำนวณราคาทีละขั้นตอน"""
    seat_extras = {"Normal": 0, "Deluxe": 40, "Honeymoon": 80, "VIP": 190}

    extra_price = seat_extras.get(booking.seat_type, 0)
    base_price = booking.showtime.base_price
    full_price = base_price + extra_price

    discount_rate = booking.customer.get_discount_rate()

    final_price = booking.net_price

    print(
        f"ราคาตั๋วภาพยนตร์เรื่อง '{booking.showtime.movie_name}' (โรง"
        f" {booking.showtime.theater_no}) = {base_price} บาท"
    )
    print(
        f"บวกเพิ่มที่นั่งประเภท '{booking.seat_type}' = +{extra_price} บาท"
        f" (ราคารวมก่อนส่วนลด = {full_price} บาท)"
    )

    if discount_rate > 0:
        print(
            f"ลูกค้าเป็นสมาชิก {booking.customer.member_type} -> ได้ส่วนลด"
            f" {discount_rate * 100:.0f}%"
        )
    else:
        print("ลูกค้าเป็นสมาชิก General -> ไม่มีส่วนลด")

    print(f"ราคาสุทธิ = {final_price:,.2f} บาท")
    return


demo_cust = Customer(
    "C001",
    f"{random.choice(first_names)} {random.choice(last_names)}",
    "VIP",
)

demo_show = Showtime(
    "ST01",
    "Avatar 3",
    1,
    datetime(2026, 8, 20, 18, 0),
    datetime(2026, 8, 20, 21, 0),
    160,
)

demo_booking = Booking("BK001", demo_cust, demo_show, "A1", "Honeymoon")
explain_booking_price(demo_booking)

ราคาตั๋วภาพยนตร์เรื่อง 'Avatar 3' (โรง 1) = 160 บาท
บวกเพิ่มที่นั่งประเภท 'Honeymoon' = +80 บาท (ราคารวมก่อนส่วนลด = 240 บาท)
ลูกค้าเป็นสมาชิก VIP -> ได้ส่วนลด 30%
ราคาสุทธิ = 168.00 บาท


## ตารางบันทึกข้อมูลที่จำลองได้

In [ ]:
# ตารางที่ 1 ตารางลูกค้า
customers_rows = [
    {
        "customer_id": b.customer.customer_id,
        "name": (
            f"{b.customer.name.split()[0]} {b.customer.name.split()[1][0]}***"
            if " " in b.customer.name
            else b.customer.name
        ),
        "member_type": b.customer.member_type,
    }
    for b in all_bookings
]

customers_df = pd.DataFrame(customers_rows).drop_duplicates(
    subset=["customer_id"]
)
customers_df.head()

,customer_id,name,member_type
0,C001,กิตติ ใ***,VIP
1,C002,ณัฐพงษ์ ม***,General
2,C003,ศิริพร ม***,Student
3,C004,วิภา ม***,General
4,C005,สมชาย ใ***,VIP


In [ ]:
# แปลงเป็น CSV
customers_df.to_csv("customers.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ตารางที่ 2 ตารางธุรกรรมการจอง
bookings_rows = [
    {
        "booking_id": b.booking_id,
        "customer_id": b.customer.customer_id,  # ใช้เป็น Foreign Key
        "movie_name": b.showtime.movie_name,
        "theater_no": b.showtime.theater_no,
        "seat_number": b.seat_number,
        "seat_type": b.seat_type,
        "price": b.net_price,
        "status": b.status,
    }
    for b in all_bookings
]

bookings_df = pd.DataFrame(bookings_rows)
bookings_df.head()

,booking_id,customer_id,movie_name,theater_no,seat_number,seat_type,price,status
0,BK001,C001,Spider-Man,2,D5,Deluxe,154.0,Paid
1,BK002,C002,Interstellar,5,F7,Normal,190.0,Paid
2,BK003,C003,Interstellar,5,F10,Deluxe,184.0,Paid
3,BK004,C004,Batman,3,B1,Honeymoon,280.0,Paid
4,BK005,C005,Inception,4,C3,Normal,119.0,Paid


In [ ]:
# แปลงเป็น CSV
bookings_df.to_csv("bookings.csv", index=False, encoding="utf-8-sig")

## ฐานข้อมูล SQLite

In [ ]:
import sqlite3
import pandas as pd

# 1. อ่านไฟล์ CSV ทั้ง 2 เข้าเป็น DataFrame
df_customers = pd.read_csv("https://raw.githubusercontent.com/natthamonsmn/Cinema_Team/refs/heads/main/customers.csv")
df_bookings = pd.read_csv("https://raw.githubusercontent.com/natthamonsmn/Cinema_Team/refs/heads/main/bookings.csv")

# 2. สร้าง connection สู่ SQLite ในหน่วยความจำชั่วคราว (:memory:)
conn = sqlite3.connect(":memory:")

# 3. นำ DataFrame ทั้ง 2 แปลงเป็นตารางใน SQLite ชั่วคราว
df_customers.to_sql("customers", conn, index=False, if_exists="replace")
df_bookings.to_sql("bookings", conn, index=False, if_exists="replace")

# 4. เขียน SQL JOIN ตาราง
query = """
SELECT
    b.booking_id,
    c.customer_id,
    c.name AS customer_name,
    c.member_type,
    b.movie_name,
    b.theater_no,
    b.seat_number,
    b.seat_type,
    b.price,
    b.status
FROM bookings b
JOIN customers c ON b.customer_id = c.customer_id
"""

# 5. ดึงข้อมูลออกมาเป็น DataFrame
df_joined = pd.read_sql_query(query, conn)

# ปิด connection
conn.close()

# แสดงผลลัพธ์
df_joined.head()

,booking_id,customer_id,customer_name,member_type,movie_name,theater_no,seat_number,seat_type,price,status
0,BK001,C001,ศิริพร ม***,Student,Avatar 3,1,C9,Deluxe,160.0,Paid
1,BK002,C002,อนันต์ ร***,VIP,Inception,4,B7,Honeymoon,0.0,Cancelled
2,BK003,C003,กิตติ ว***,Student,Interstellar,5,C3,Honeymoon,0.0,Cancelled
3,BK004,C004,ศิริพร ส***,Student,Avatar 3,1,B7,Honeymoon,192.0,Paid
4,BK005,C005,ชลธิชา เ***,VIP,Spider-Man,2,D6,Normal,126.0,Paid


In [ ]:
# แปลงเป็น CSV
df_joined.to_csv("joined.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ตารางฉบับเต็ม (ลูกค้า + ธุรกรรมการจอง)
pd.read_csv("https://raw.githubusercontent.com/natthamonsmn/Cinema_Team/refs/heads/main/joined.csv")

,booking_id,customer_id,customer_name,member_type,movie_name,theater_no,seat_number,seat_type,price,status
0,BK001,C001,ศิริพร ม***,Student,Avatar 3,1,C9,Deluxe,160.0,Paid
1,BK002,C002,อนันต์ ร***,VIP,Inception,4,B7,Honeymoon,0.0,Cancelled
2,BK003,C003,กิตติ ว***,Student,Interstellar,5,C3,Honeymoon,0.0,Cancelled
3,BK004,C004,ศิริพร ส***,Student,Avatar 3,1,B7,Honeymoon,192.0,Paid
4,BK005,C005,ชลธิชา เ***,VIP,Spider-Man,2,D6,Normal,126.0,Paid
...,...,...,...,...,...,...,...,...,...,...
295,BK296,C296,ธนกฤต ส***,Student,Inception,4,D10,Deluxe,168.0,Paid
296,BK297,C297,สมชาย เ***,Student,Interstellar,5,F5,VIP,0.0,Rejected (ที่นั่ง F5 ถูกจองแล้ว (Double Booking))
297,BK298,C298,อนันต์ ส***,Student,Inception,4,F8,Normal,136.0,Paid
298,BK299,C299,กมล ร***,Student,Batman,3,A8,VIP,0.0,Rejected (ที่นั่ง A8 ถูกจองแล้ว (Double Booking))


###ส่วนที่ 4 ระบบบการจอง

In [ ]:
def simulate_customer_visit(
    customer,
    showtime,
    seat_number,
    seat_type,
    existing_seats,
    promo_discount=0.0,
):

    print("เริ่มกระบวนการจองตั๋วภาพยนตร์")

    masked_name = (
        f"{customer.name.split()[0]} {customer.name.split()[1][0]}***"
        if " " in customer.name
        else customer.name
    )

    # Step 1 ลูกค้าเข้ามา
    print("\n ลูกค้าเดินเข้ามาจองตั๋ว")
    print(f"รหัสลูกค้า: {customer.customer_id}")
    print(f"ชื่อลูกค้า: {masked_name}")
    print(f"ประเภทสมาชิก: {customer.member_type}")

    # Step 2 เลือกรอบหนัง
    print("\n เลือกรอบภาพยนตร์")
    print(f"ภาพยนตร์: {showtime.movie_name}")
    print(f"โรงภาพยนตร์: {showtime.theater_no}")
    print(f"เวลาเริ่ม: {showtime.start_time}")

    # Step 3 ตรวจสอบที่นั่ง
    print("\n ตรวจสอบที่นั่ง")

    is_valid, seat_msg = validate_seat_booking(seat_number, existing_seats)

    print(seat_msg)

    if not is_valid:
        print("\nไม่สามารถทำรายการจองได้")
        return None

    # Step 4 คำนวณราคา
    print("\n คำนวณราคาตั๋ว")

    seat_extras = {"Normal": 0, "Deluxe": 40, "Honeymoon": 80, "VIP": 190}

    # ราคาปกติ = ราคาตั๋วพื้นฐาน + ค่าเพิ่มประเภทที่นั่ง
    normal_price = showtime.base_price + seat_extras.get(seat_type, 0)

    # ราคาสุทธิหลังหักส่วนลด
    net_price = calculate_booking_price(
        showtime.base_price, seat_type, customer.member_type, promo_discount
    )

    print(f"ประเภทที่นั่ง: {seat_type}")
    print(f"ราคาปกติ: {normal_price} บาท")
    print(f"ราคาสุทธิ: {net_price} บาท")

    # Step 5 สร้าง Booking
    print("\n สร้างรายการจอง")

    booking_id = f"BK{customer.customer_id}"

    booking = Booking(
        booking_id, customer, showtime, seat_number, seat_type
    )

    # ใช้ราคาที่คำนวณจากโปรโมชั่น
    booking.net_price = net_price

    print("สร้างรายการจองสำเร็จ")
    print(f"Booking ID: {booking.booking_id}")
    print(f"สถานะ: {booking.status}")

    # Step 6 ชำระเงิน
    print("\n ลูกค้าชำระเงิน")

    try:
        paid_price = booking.confirm_payment()

        print(f"ชำระเงินสำเร็จ: {paid_price} บาท")
        print(f"สถานะปัจจุบัน: {booking.status}")

        # เพิ่มที่นั่งเข้าไปในรายการที่จองแล้ว
        existing_seats.append(seat_number.upper())

    except Exception as e:
        print(f"ชำระเงินไม่สำเร็จ: {e}")
        return booking

    # Step 7 สะสมคะแนน
    print("\n สะสมคะแนนสมาชิก")
    print(f"คะแนนสะสมปัจจุบัน: {customer.points} คะแนน")

    # Step 8 สแกนตั๋ว
    print("\n สแกนตั๋วเข้าโรงภาพยนตร์")

    success, scan_msg = process_ticket_scan(booking.booking_id, booking.status)

    print(scan_msg)

    # สรุปผล
    print("\nสรุปการใช้บริการ")

    print(f"ลูกค้า: {masked_name}")
    print(f"ภาพยนตร์: {showtime.movie_name}")
    print(f"โรงภาพยนตร์: {showtime.theater_no}")
    print(f"ที่นั่ง: {booking.seat_number}")
    print(f"ประเภทที่นั่ง: {booking.seat_type}")
    print(f"ราคาปกติ: {normal_price} บาท")
    print(f"ราคาสุทธิ: {booking.net_price} บาท")
    print(f"สถานะการจอง: {booking.status}")
    print(f"คะแนนสะสม: {customer.points} คะแนน")

    return booking

In [ ]:
customer_1 = Customer(
    customer_id="C001",
    name=f"{random.choice(first_names)} {random.choice(last_names)}",
    member_type="Student",
)

showtime_1 = Showtime(
    "ST01",
    "Avatar 3",
    1,
    datetime(2026, 8, 20, 14, 0),
    datetime(2026, 8, 20, 17, 0),
    160,
)

existing_seats = ["A1", "A2", "B5"]

# รันฟังก์ชันจำลองการจองตั๋ว
booking_result = simulate_customer_visit(
    customer=customer_1,
    showtime=showtime_1,
    seat_number="C3",
    seat_type="Deluxe",
    existing_seats=existing_seats,
    promo_discount=0.10,
)

เริ่มกระบวนการจองตั๋วภาพยนตร์

 ลูกค้าเดินเข้ามาจองตั๋ว
รหัสลูกค้า: C001
ชื่อลูกค้า: วิภา ว***
ประเภทสมาชิก: Student

 เลือกรอบภาพยนตร์
ภาพยนตร์: Avatar 3
โรงภาพยนตร์: 1
เวลาเริ่ม: 2026-08-20 14:00:00

 ตรวจสอบที่นั่ง
ที่นั่งว่าง สามารถจองได้

 คำนวณราคาตั๋ว
ประเภทที่นั่ง: Deluxe
ราคาปกติ: 200 บาท
ราคาสุทธิ: 160.0 บาท

 สร้างรายการจอง
สร้างรายการจองสำเร็จ
Booking ID: BKC001
สถานะ: Pending

 ลูกค้าชำระเงิน
ชำระเงินสำเร็จ: 160.0 บาท
สถานะปัจจุบัน: Paid

 สะสมคะแนนสมาชิก
คะแนนสะสมปัจจุบัน: 16 คะแนน

 สแกนตั๋วเข้าโรงภาพยนตร์
สแกนสำเร็จ! (Used)

สรุปการใช้บริการ
ลูกค้า: วิภา ว***
ภาพยนตร์: Avatar 3
โรงภาพยนตร์: 1
ที่นั่ง: C3
ประเภทที่นั่ง: Deluxe
ราคาปกติ: 200 บาท
ราคาสุทธิ: 160.0 บาท
สถานะการจอง: Paid
คะแนนสะสม: 16 คะแนน


### ส่วนที่ 5

In [ ]:
# 1. สร้างรอบหนังครบทั้ง 5 เรื่อง (5 โรง)
showtime_1 = Showtime(
    "ST01",
    "Avatar 3",
    1,
    datetime(2026, 8, 20, 14, 0),
    datetime(2026, 8, 20, 17, 0),
    160,
)
showtime_2 = Showtime(
    "ST02",
    "Spider-Man",
    2,
    datetime(2026, 8, 20, 15, 30),
    datetime(2026, 8, 20, 18, 0),
    180,
)
showtime_3 = Showtime(
    "ST03",
    "Batman",
    3,
    datetime(2026, 8, 20, 19, 0),
    datetime(2026, 8, 20, 22, 0),
    200,
)
showtime_4 = Showtime(
    "ST04",
    "Inception",
    4,
    datetime(2026, 8, 20, 13, 0),
    datetime(2026, 8, 20, 15, 30),
    170,
)
showtime_5 = Showtime(
    "ST05",
    "Interstellar",
    5,
    datetime(2026, 8, 20, 20, 0),
    datetime(2026, 8, 20, 23, 0),
    190,
)

# รวมรอบหนังทั้ง 5 เรื่องไว้สำหรับสุ่มเลือก
showtimes_pool = [
    showtime_1,
    showtime_2,
    showtime_3,
    showtime_4,
    showtime_5,
]

# เก็บที่นั่งที่ถูกจองแยกตามโรง (รองรับโรง 1 ถึง 5)
booked_seats_by_theater = {1: [], 2: [], 3: [], 4: [], 5: []}

# ข้อมูลสำหรับสุ่ม
member_pool = ["General", "Student", "VIP"]
seat_type_pool = ["Normal", "Deluxe", "Honeymoon", "VIP"]
seat_pool = [
    f"{row}{num}" for row in "ABCDEF" for num in range(1, 11)
] + ["Z99"]

# 2. ลูปสุ่มจำลองลูกค้า
for i in range(1, 6):
    print(f"\nลูกค้าคนที่ {i}")

    # สุ่มข้อมูลต่างๆ
    chosen_member = random.choice(member_pool)
    chosen_showtime = random.choice(showtimes_pool)
    chosen_seat = random.choice(seat_pool)
    chosen_seat_type = random.choice(seat_type_pool)

    # สุ่มชื่อ-นามสกุลจริงให้ลูกค้า
    random_fullname = (
        f"{random.choice(first_names)} {random.choice(last_names)}"
    )

    # สร้าง Object Customer
    customer = Customer(
        customer_id=f"C{i:03d}",
        name=random_fullname,
        member_type=chosen_member,
    )

    # เรียกใช้ฟังก์ชันจำลองการจอง
    booking_result = simulate_customer_visit(
        customer=customer,
        showtime=chosen_showtime,
        seat_number=chosen_seat,
        seat_type=chosen_seat_type,
        # ส่งรายการที่นั่งแยกตามโรงที่เลือก
        existing_seats=booked_seats_by_theater[chosen_showtime.theater_no],
        # สุ่มส่วนลดโปรโมชัน 0% หรือ 10%
        promo_discount=random.choice([0.0, 0.10]),
    )


ลูกค้าคนที่ 1
เริ่มกระบวนการจองตั๋วภาพยนตร์

 ลูกค้าเดินเข้ามาจองตั๋ว
รหัสลูกค้า: C001
ชื่อลูกค้า: กมล ม***
ประเภทสมาชิก: General

 เลือกรอบภาพยนตร์
ภาพยนตร์: Inception
โรงภาพยนตร์: 4
เวลาเริ่ม: 2026-08-20 13:00:00

 ตรวจสอบที่นั่ง
ที่นั่งว่าง สามารถจองได้

 คำนวณราคาตั๋ว
ประเภทที่นั่ง: Deluxe
ราคาปกติ: 210 บาท
ราคาสุทธิ: 210.0 บาท

 สร้างรายการจอง
สร้างรายการจองสำเร็จ
Booking ID: BKC001
สถานะ: Pending

 ลูกค้าชำระเงิน
ชำระเงินสำเร็จ: 210.0 บาท
สถานะปัจจุบัน: Paid

 สะสมคะแนนสมาชิก
คะแนนสะสมปัจจุบัน: 21 คะแนน

 สแกนตั๋วเข้าโรงภาพยนตร์
สแกนสำเร็จ! (Used)

สรุปการใช้บริการ
ลูกค้า: กมล ม***
ภาพยนตร์: Inception
โรงภาพยนตร์: 4
ที่นั่ง: B4
ประเภทที่นั่ง: Deluxe
ราคาปกติ: 210 บาท
ราคาสุทธิ: 210.0 บาท
สถานะการจอง: Paid
คะแนนสะสม: 21 คะแนน

ลูกค้าคนที่ 2
เริ่มกระบวนการจองตั๋วภาพยนตร์

 ลูกค้าเดินเข้ามาจองตั๋ว
รหัสลูกค้า: C002
ชื่อลูกค้า: ณัฐพงษ์ ม***
ประเภทสมาชิก: Student

 เลือกรอบภาพยนตร์
ภาพยนตร์: Spider-Man
โรงภาพยนตร์: 2
เวลาเริ่ม: 2026-08-20 15:30:00

 ตรวจสอบที่นั่ง
ที่นั่งว่าง สามารถจอ